# MLflow Tracing Quickstart
from https://gist.github.com/djliden/bfb667b35df323d3a70c080f89ab1edc#file-01_tracing-ipynb

### Without Tracing

```
RunResult:
- Last agent: Agent(name="Product Specialist", ...)
- Final output (str):
    I’m unable to retrieve the details for product ID p120 at the moment. Could you please double-check the product ID, or let me know if there is anything else I can assist you with?
- 8 new item(s)
- 4 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
```

### With Tracing

![](https://gist.githubusercontent.com/djliden/bfb667b35df323d3a70c080f89ab1edc/raw/0012212817e98a43fe117d203c9082cc6b9496d3/trace_err.png)

## Setup

1. Install dependencies
```bash
pip install mlflow openai anthropic python-dotenv openai-agents
```

2. [Get an OpenAI API key](https://platform.openai.com/docs/libraries#create-and-export-an-api-key)

3. Set the OpenAI API key as an environment variable
```bash
export OPENAI_API_KEY=<your-openai-api-key>
```


## MLflow Autologging for Automatic Tracing

In [1]:
import mlflow
from openai import OpenAI

### Without Tracing

In [2]:
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "What is an MLflow tracking server?"
        }
    ],
    temperature=0.5
)

print(completion)

ChatCompletion(id='chatcmpl-CwyRdyrW8vyHHxTep25bdRQyuMcmz', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='An MLflow Tracking Server is a component of the MLflow platform that is used to log and manage machine learning experiments. It provides a centralized way to track the parameters, metrics, and artifacts associated with your machine learning runs. Here’s a breakdown of its key features and functionalities:\n\n1. **Experiment Tracking**: The tracking server allows users to log various aspects of their machine learning experiments, such as hyperparameters, metrics (like accuracy, loss), and artifacts (like model files, plots, etc.). This helps in comparing different runs and understanding the impact of different configurations.\n\n2. **Centralized Storage**: By using a tracking server, you can store all your experiment data in a centralized location. This is particularly useful for teams working collaboratively, as it allows every

### With Tracing

In [3]:
mlflow.openai.autolog()

# optional, but recommended
mlflow.set_experiment("openai-example")

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "What is an MLflow tracking server?"
        }
    ]
)

print(completion)

2026/01/11 23:43:45 INFO mlflow.tracking.fluent: Experiment with name 'openai-example' does not exist. Creating a new experiment.


ChatCompletion(id='chatcmpl-CwyRtGkOtDnUyJpCU8trRDjEo2Cs4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='An MLflow Tracking Server is a component of the MLflow platform that provides a way to log and query experimental results of machine learning workflows. It enables data scientists and machine learning engineers to keep track of their experiments and model training processes in an organized manner. Here are the key features and functions of an MLflow Tracking Server:\n\n1. **Experiment Management**: The tracking server allows users to create and manage multiple experiments, making it easier to organize runs related to different projects or objectives.\n\n2. **Logging**: Users can log various parameters, metrics, and artifacts during their machine learning runs. This includes:\n   - **Parameters**: Hyperparameters or configurations used during the training.\n   - **Metrics**: Performance metrics like accuracy, loss, etc., evaluat

Trace(trace_id=tr-aa4bed91349bcc59de9caa89546f114d)

### Open the MLflow UI to see the trace

```bash
mlflow ui
```

### Preview traces in the notebook

In [ ]:
# set the tracking URI to the local MLflow server
mlflow.set_tracking_uri("http://127.0.0.1:5001")

from pydantic import BaseModel
from openai import OpenAI

client = OpenAI()

class EventDetails(BaseModel):
    speakers: str
    date: str
    num_attendees: int
    description: str

raw_event_details = """137 views  Streamed live on Mar 6, 2025
Stop fighting with model signatures and focus on your model logic! MLflow 2.20.0 lets you use Python's native type annotations to automatically validate inputs and infer model signatures. MLflow Software Engineer, Serena Ruan, will show you how this addresses some of the most common pain points in custom model development and simplifies your workflow with patterns for everything from simple data types to complex Pydantic models.

Come see how these improvements make your MLflow experience more Pythonic and robust, with fewer runtime surprises. Bring your questions for our Q&A session!"""

completion = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {"role": "user", "content": raw_event_details},
    ],
    response_format=EventDetails,
)

print(completion)

In [ ]:
# mlflow.tracing.disable_notebook_display()

### Organizing Traces

In [ ]:
with mlflow.start_run() as run:
    raw_event_details_1 = """137 views  Streamed live on Mar 6, 2025
Stop fighting with model signatures and focus on your model logic! MLflow 2.20.0 lets you use Python's native type annotations to automatically validate inputs and infer model signatures. MLflow Software Engineer, Serena Ruan, will show you how this addresses some of the most common pain points in custom model development and simplifies your workflow with patterns for everything from simple data types to complex Pydantic models.

Come see how these improvements make your MLflow experience more Pythonic and robust, with fewer runtime surprises. Bring your questions for our Q&A session!"""

    raw_event_details_2 = """40 views  
Streamed live Mar 26, 2025
This community meetup will focus on 2️⃣ major updates:

🚀 MLflow Prompt Registry – As large language models (LLMs) become integral to AI workflows, MLflow is expanding its capabilities to support prompt engineering. Learn how the new MLflow Prompt Registry enables seamless tracking, versioning, and experimentation with prompts—making it easier to manage LLM-based applications.

📦 MLflow 3: A Model-Centric Approach – MLflow 3 introduces a more structured, model-first approach to managing the ML lifecycle. This new design enhances how models are tracked, stored, and deployed, making workflows more intuitive and scalable. Learn what’s changing and how these improvements will help teams streamline their machine learning operations.

Harutaka Kawamura and Yuki Watanabe will walk through these latest developments and answer your MLflow questions in a live Q&A session!"""

    for e in [raw_event_details_1, raw_event_details_2]:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "Extract the event information."},
                {"role": "user", "content": e},
            ],
            response_format=EventDetails,
        )

        print(completion)



## Tracing Other Providers

![](https://gist.githubusercontent.com/djliden/bfb667b35df323d3a70c080f89ab1edc/raw/0012212817e98a43fe117d203c9082cc6b9496d3/providers.png)

For more information, see the [MLflow Tracing documentation](https://mlflow.org/docs/latest/tracing/#automatic-tracing).

### Anthropic example

In [ ]:
import anthropic
from dotenv import load_dotenv

# load_dotenv()

mlflow.anthropic.autolog()

client = anthropic.Anthropic()

message = client.messages.create(
    model="claude-3-7-sonnet-20250219",
    max_tokens=1000,
    temperature=1,
    messages=[
        {
            "role": "user",
            "content": "What is an MLflow tracking server?"
        }
    ]
)

## More Complex Tracing: OpenAI Agents SDK

In [ ]:
import mlflow
import asyncio
from agents import Agent, Runner, function_tool

mlflow.openai.autolog()

mlflow.set_experiment("Customer Support Demo")

# Define our tool for product lookup
@function_tool
def lookup_product_details(product_id: str) -> str:
    """Look up details for a product by ID."""
    products = {
        "p100": "Basic Plan: $19/month, 1000 API calls",
        "p200": "Pro Plan: $49/month, unlimited API calls",
        "p300": "Enterprise: $499/month, dedicated support"
    }
    # return products[product_id]
    return products.get(product_id, "Product not found. Inform the customer that the available products are: p100, p200, p300")

# Create specialized agents
product_agent = Agent(
    name="Product Specialist",
    instructions="You help customers with product information. Use the lookup_product_details tool when needed.",
    tools=[lookup_product_details],
)

technical_agent = Agent(
    name="Technical Support",
    instructions="You help customers with API connection issues and technical problems.",
)

# Create triage agent that can hand off to specialized agents
triage_agent = Agent(
    name="Support Triage",
    instructions="""
    You are the initial point of contact. Determine if the customer needs:
    1. Product information: Hand off to the Product Specialist
    2. Technical support: Hand off to Technical Support
    
    Be concise in your response before handing off.
    """,
    handoffs=[product_agent, technical_agent],
)

async def main():
    # This query should trigger a product lookup
    query = "What features are included in the Plan? I think the product ID is p120."
    
    result = await Runner.run(triage_agent, input=query)
    print(result)

await main()